# Tokenisers — BPE vs WordPiece vs SentencePiece

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

Subword tokenisers split text into pieces that balance vocabulary size and out-of-vocabulary handling. BPE merges the most frequent adjacent pair iteratively; WordPiece picks the merge that maximises corpus likelihood; SentencePiece operates directly on raw bytes/UTF-8 (no whitespace assumption).


## Mathematical Formulation

BPE picks merges $(a, b) \to ab$ that maximise frequency:

$$(a^*, b^*) = \arg\max_{(a, b)} \text{count}(ab \mid \text{corpus})$$

WordPiece picks merges that maximise the *score*

$$\text{score}(a, b) = \frac{\text{count}(ab)}{\text{count}(a)\cdot \text{count}(b)}$$


## Implementation


In [ ]:
# pip install tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace


In [ ]:
corpus = [
    'the quick brown fox jumps over the lazy dog',
    'jumping cats and lazy dogs',
    'transformers tokenise subwords efficiently',
] * 50  # tiny corpus, repeated

def train_bpe(corpus, vocab_size=80):
    tok = Tokenizer(BPE(unk_token='[UNK]'))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=['[UNK]'])
    tok.train_from_iterator(corpus, trainer)
    return tok

def train_wp(corpus, vocab_size=80):
    tok = Tokenizer(WordPiece(unk_token='[UNK]'))
    tok.pre_tokenizer = Whitespace()
    trainer = WordPieceTrainer(vocab_size=vocab_size, special_tokens=['[UNK]'])
    tok.train_from_iterator(corpus, trainer)
    return tok


## Experiment


In [ ]:
bpe = train_bpe(corpus)
wp = train_wp(corpus)

sentence = 'jumping subwords tokenise efficiently'
print('BPE      :', bpe.encode(sentence).tokens)
print('WordPiece:', wp.encode(sentence).tokens)


## Discussion

- BPE is greedy on frequency; WordPiece's score is closer to a likelihood criterion and handles morphology slightly better.
- SentencePiece (Unigram or BPE) operates at the byte level and is whitespace-agnostic — preferred for non-English / multilingual.
- Vocab size is a tunable hyperparameter; common ranges are 16k–64k for monolingual, 200k+ for multilingual.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
